# Blue Carbon Investment Widget data preparation. 

The final data needs to follow this data model:

- id (int)  
- location_id (str)  
- category (str)  
- area (numeric)  
- text (str)



In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Read dataframe from GCS

In [16]:
carbon_data = pd.read_csv('https://storage.googleapis.com/mangrove_atlas/widget_data/carbon_investment.csv')

metadata_carbon = {'ISO':carbon_data.columns[0] , 'Country':carbon_data.columns[1], 'Text_Carbon_5':carbon_data.columns[2], 'Area_Carbon_5':carbon_data.columns[3],
                    'Text_Carbon_10':carbon_data.columns[4], 'Area_Carbon_10':carbon_data.columns[5],'Diff_carbon_5_10':carbon_data.columns[6], 'Area_Protected':carbon_data.columns[7], 'Area_Remaining':carbon_data.columns[8]}
carbon_data.columns = ['ISO', 'Country', 'Text_Carbon_5', 'Area_Carbon_5', 'Text_Carbon_10', 'Area_Carbon_10','Diff_carbon_5_10','Area_Protected', 'Area_Remaining']
carbon_data['Area_Carbon_5'] = carbon_data['Area_Carbon_5'].str.replace(',', '').astype(np.float64)
carbon_data['Area_Carbon_10'] = carbon_data['Area_Carbon_10'].str.replace(',', '').astype(np.float64)
carbon_data['Area_Remaining'] = carbon_data['Area_Remaining'].str.replace(',', '').astype(np.float64)
#carbon_data.set_index('ISO', inplace = True)
carbon_data

,ISO,Country,Text_Carbon_5,Area_Carbon_5,Text_Carbon_10,Area_Carbon_10,Diff_carbon_5_10,Area_Protected,Area_Remaining
0,AGO,Angola,900 (±100),900.0,"1,300 (± 100)",1300.0,400,168.7598,35058.97
1,ATG,Antigua & Barbuda,0.00,0.0,0.00,0.0,0,444.4700,445.21
2,AUS,Australia,0.00,0.0,0.00,0.0,0,484299.6015,482962.98
3,BHR,Bahrain,0.00,0.0,0.00,0.0,0,0.0000,82.03
4,BGD,Bangladesh,"98,900 (±2,800)",98900.0,"113,700 (± 3,200)",113700.0,14800,375201.7996,0.00
...,...,...,...,...,...,...,...,...,...
96,VUT,Vanuatu,0,0.0,0,0.0,0,0.0000,1769.02
97,VEN,Venezuela,"3,000 (±700)",3000.0,"8,400 (± 2,000)",8400.0,5400,185790.9775,85414.87
98,VNM,Vietnam,"75,900 (±3,200)",75900.0,"90,900 (± 3,800)",90900.0,15000,73507.8649,0.00
99,VGB,"Virgin Islands, British",0,0.0,0,0.0,0,0.0000,87.21


## 2. Load location_id data and join

In [30]:
loc = pd.read_csv('https://storage.googleapis.com/mangrove_atlas/widget_data/Country_location_id.csv', )
loc = loc[['ISO', 'location_id']]
loc

,ISO,location_id
0,AGO,1_2_97
1,ATG,1_2_69
2,AUS,1_2_98
3,BHR,1_2_73
4,BGD,1_2_72
...,...,...
97,VUT,1_2_93
98,VEN,1_2_61
99,VNM,1_2_63
100,VGB,1_2_62


In [32]:
df_join = loc.join(carbon_data.set_index('ISO'), on='ISO', how = 'inner')
df_join.drop(columns=['Diff_carbon_5_10'], inplace=True)
df_join

,ISO,location_id,Country,Text_Carbon_5,Area_Carbon_5,Text_Carbon_10,Area_Carbon_10,Area_Protected,Area_Remaining
0,AGO,1_2_97,Angola,900 (±100),900.0,"1,300 (± 100)",1300.0,168.7598,35058.97
1,ATG,1_2_69,Antigua & Barbuda,0.00,0.0,0.00,0.0,444.4700,445.21
2,AUS,1_2_98,Australia,0.00,0.0,0.00,0.0,484299.6015,482962.98
3,BHR,1_2_73,Bahrain,0.00,0.0,0.00,0.0,0.0000,82.03
4,BGD,1_2_72,Bangladesh,"98,900 (±2,800)",98900.0,"113,700 (± 3,200)",113700.0,375201.7996,0.00
...,...,...,...,...,...,...,...,...,...
97,VUT,1_2_93,Vanuatu,0,0.0,0,0.0,0.0000,1769.02
98,VEN,1_2_61,Venezuela,"3,000 (±700)",3000.0,"8,400 (± 2,000)",8400.0,185790.9775,85414.87
99,VNM,1_2_63,Vietnam,"75,900 (±3,200)",75900.0,"90,900 (± 3,800)",90900.0,73507.8649,0.00
100,VGB,1_2_62,"Virgin Islands, British",0,0.0,0,0.0,0.0000,87.21


## 3. Change from wide to liong format and combine area and text data

In [45]:
text_col = ['Text_Carbon_5','Text_Carbon_10']
area_col = ['Area_Carbon_5','Area_Carbon_10','Area_Protected','Area_Remaining']

df_area = pd.melt(df_join.drop(columns=text_col),
    id_vars=['ISO', 'Country', 'location_id'],
    var_name='category',
    value_name='area')
df_area['category'] = df_area['category'].str.replace("Area_", "")

df_text = pd.melt(df_join.drop(columns=area_col),
    id_vars=['ISO', 'Country', 'location_id'],
    var_name='category',
    value_name='text')
df_text['category'] = df_text['category'].str.replace("Text_", "")



df_out = pd.merge(df_area, df_text, on = ['ISO','location_id','Country', 'category'], how='left')
df_out

,ISO,Country,location_id,category,area,text
0,AGO,Angola,1_2_97,Carbon_5,900.00,900 (±100)
1,ATG,Antigua & Barbuda,1_2_69,Carbon_5,0.00,0.00
2,AUS,Australia,1_2_98,Carbon_5,0.00,0.00
3,BHR,Bahrain,1_2_73,Carbon_5,0.00,0.00
4,BGD,Bangladesh,1_2_72,Carbon_5,98900.00,"98,900 (±2,800)"
...,...,...,...,...,...,...
399,VUT,Vanuatu,1_2_93,Remaining,1769.02,NaN
400,VEN,Venezuela,1_2_61,Remaining,85414.87,NaN
401,VNM,Vietnam,1_2_63,Remaining,0.00,NaN
402,VGB,"Virgin Islands, British",1_2_62,Remaining,87.21,NaN


## 4. Upload to API